In [14]:
from google.colab import files
files.upload()

In [15]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [16]:
!kaggle datasets download -d abhinavkulshreshth/pothole-detection-dataset

Dataset URL: https://www.kaggle.com/datasets/abhinavkulshreshth/pothole-detection-dataset
License(s): CC0-1.0
pothole-detection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [17]:
!unzip -qq pothole-detection-dataset.zip -d dataset

replace dataset/Dataset/test/normal0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace dataset/Dataset/test/normal0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace dataset/Dataset/test/normal1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [18]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### Data Loading

In [19]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2,contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


In [20]:
traindataset = datasets.ImageFolder('/content/dataset/Dataset/train',transform=transform)
valdataset = datasets.ImageFolder('/content/dataset/Dataset/val',transform=transform)

In [21]:
trainloader = DataLoader(traindataset,batch_size=32,shuffle=True,num_workers=2)
valloader = DataLoader(valdataset, batch_size=32, shuffle=False,num_workers=2)

In [22]:
classnames = traindataset.classes
print('Classes : ',classnames)

Classes :  ['Normal', 'Pothole']


### Transfer Learning

In [23]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(classnames))

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=3,gamma=0.1)


### Training

In [26]:
num_epochs=8
best_acc = 0.0

In [ ]:
for epoch in range(num_epochs):
  model.train()
  running_loss=0.0

  for inputs , labels in trainloader:
    inputs ,labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()

  model.eval()
  correct=0
  total=0
  with torch.no_grad():
    for inputs , labels in valloader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)
      _, predicted = torch.max(outputs,1)
      total+=labels.size(0)
      correct+=(predicted == labels).sum().item()

  acc = 100*correct/total
  print(f'Epoch {epoch+1}/{num_epochs} | Loss : {running_loss:.4f}  | Val Acc: {acc:.2f}')

  if acc > best_acc:
    best_acc = acc
    torch.save(model.state_dict(), 'guard_resnet18.pth')
    print('Best Model saved')

print('Training Finished!')
print('Accuracy  : ', best_acc)
